In [ ]:
from keys import secret, key, url

In [ ]:
import alpaca_trade_api as api
import alpaca_trade_api.rest as rapi

In [ ]:
import pandas as pd

In [ ]:
rest = api.REST(key, secret, url)

In [ ]:
bars = rest.get_bars("spy", rapi.TimeFrame.Minute, "2021-01-01", "2021-12-31")

bars_ = [bar._raw for bar in bars]

x = pd.DataFrame(bars_)

In [ ]:
x

In [ ]:
x.to_csv("spy-2021.csv")

In [ ]:
prev = 10
columns = x.drop("t", axis=1).columns
groups = [x, ] + [x.drop("t", axis=1).shift(1).rename(lambda x: x+f"_{n}", axis=1) for n in range(1, prev+1) ]
xx = pd.concat(groups, axis=1)

for n in range(1, prev):
    for col in columns:
        xx[f"{col}_{n}_delta"] = (xx[f"{col}_{n}"] / xx[f"{col}_{n+1}"])-1

for col in columns:
    xx[f"{col}_0_delta"] = (xx[col] / xx[f"{col}_{1}"])-1
xx.dropna(inplace=True)

### Instructions
Each row contains o/h/l/c/v/n along with the previous N values. 

Hence, when training a model, drop all of the "current" values since those are the target values.

In [ ]:
xx

In [ ]:
xx.delta.plot.hist(bins=100, log=True)

In [ ]:
# Imports
import tensorflow
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical

In [ ]:


# Configuration options
feature_vector_length = 784
num_classes = 10

# Load the data
(X_train, Y_train), (X_test, Y_test) = mnist.load_data()

# Reshape the data - MLPs do not understand such things as '2D'.
# Reshape to 28 x 28 pixels = 784 features
X_train = X_train.reshape(X_train.shape[0], feature_vector_length)
X_test = X_test.reshape(X_test.shape[0], feature_vector_length)

# Convert into greyscale
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')
X_train /= 255
X_test /= 255

# Convert target classes to categorical ones
Y_train = to_categorical(Y_train, num_classes)
Y_test = to_categorical(Y_test, num_classes)


In [ ]:
target_columns = [
    't', 'o', 'h', 'l', 'c', 'v', 'n', 'vw','o_0_delta', 'h_0_delta',
    'l_0_delta', 'c_0_delta', 'v_0_delta', 'n_0_delta', 'vw_0_delta'
]
targets = xx[target_columns]
train = xx.drop(target_columns, axis=1)

In [ ]:
import numpy as np

In [ ]:
from sklearn.preprocessing import LabelBinarizer

labeler = LabelBinarizer()


In [ ]:

target_train = targets[:150000]
target_test = targets[150000:]

In [ ]:
target_col = "c_0_delta"

In [ ]:
y_train = labeler.fit_transform(np.clip(target_train[target_col]*10000, -20, 20).astype(int)+20)
y_test = labeler.fit_transform(np.clip(target_test[target_col]*10000, -20, 20).astype(int)+20)

In [ ]:
y_train = target_train[target_col]>0
y_test = target_test[target_col]>0

In [ ]:
y_train

In [ ]:
y_train.shape

In [ ]:
x_train = train[:150000]
x_test = train[150000:]

# y_train = np.clip(target_train["h_0_delta"]*10000, -20, 20).astype(int)+20
# y_test = np.clip(target_test["h_0_delta"]*10000, -20, 20).astype(int)+20

In [ ]:
train_width = len(x_train.columns)

In [ ]:
targets

In [ ]:


# Set the input shape
input_shape = (train_width,)
print(f'Feature shape: {input_shape}')

# Create the model
model = Sequential()
model.add(Dense(350, input_shape=input_shape, activation='sigmoid'))
model.add(Dense(500, activation='sigmoid'))
model.add(Dense(500, activation='sigmoid'))
model.add(Dense(500, activation='sigmoid'))
model.add(Dense(1, activation='softmax'))

# Configure the model and start training
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(x_train, y_train, epochs=10, batch_size=10000, verbose=1, validation_split=0.2)

# Test the model after training
test_results = model.evaluate(x_test, y_test, verbose=1)
print(f'Test results - Loss: {test_results[0]} - Accuracy: {test_results[1]}%')

In [ ]:
y_train.argmax(axis=1)

In [ ]:

from numpy import loadtxt
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [ ]:
model = XGBClassifier(max_depth=10)
model.fit(x_train, y_train)

In [ ]:
y_pred = model.predict(x_test)

In [ ]:

# evaluate predictions
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: %.2f%%" % (accuracy * 100.0))

In [ ]:
y_train.hist(bins=np.arange(-.002, .002, .00001), log=True, )

In [ ]:
np.around(.00016*100000)